# Advection-Diffusion Model Exploration with Operator Composition

This notebook allows you to:
- Explore your DISCO model trained on combined physics equations (EULER, HEAT, DISP)
- Test operator composition methods (Greedy, Random, Exhaustive)
- Compare performance on out-of-distribution data
- Analyze operator usage patterns across physics types

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import h5py
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from torch.utils.data import DataLoader
import random
from itertools import permutations
import math
import time

# Add project root to path
sys.path.append('/mnt/home/lserrano/disco-ball')
sys.path.append('/mnt/home/lserrano/disco-ball/tests/neural-operator-splitting')

from train.train import DISCOLitModule, TemporalBatchDatasetFly # HDF5TemporalDataset
from src.utils.database import RelativeL2
from src.operators.disco import DISCOHouse
from operator_utils import sequential_operator_composition, strang_splitting_composition
from einops import rearrange

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Configuration

In [ ]:
# Model configuration - UPDATE THESE PATHS
run_name = "DISCO_advection-diffusion_solverrk4_adjFalse_h128_t2_steps1_initTrue_bs64_lr0.0005_ctxTrue_noise0_inframes16_outframes16_T10"
MODEL_CHECKPOINT_PATH = f"/mnt/home/lserrano/disco-ball/outputs/{run_name}/last-v1.ckpt"
# Data paths

# Test configuration
BATCH_SIZE = 64
N_INPUT_FRAMES = 16
N_OUTPUT_FRAMES = 34
SUB_X = 1
SUB_T = 1 #1

# Operator composition configuration
OPERATOR_CONFIG = {
    'num_operators': 64,  # Number of operators to encode
    'n_trajectories_per_operator': 1,  # Trajectories per operator (anti-forgetting)
    'max_operators': 5,  # Maximum operators in composition
    'min_improvement_threshold': 5.0,  # Minimum improvement % to add operator
    'n_input_frames': N_INPUT_FRAMES,
    'n_output_frames': N_OUTPUT_FRAMES
}

print("Configuration:")
print(f"  Model path: {MODEL_CHECKPOINT_PATH}")
print(f"  Operator config: {OPERATOR_CONFIG}")

## Load Model and Data

In [ ]:
def load_model_from_checkpoint(checkpoint_path):
    """Load DISCO model from Lightning checkpoint"""
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint not found: {checkpoint_path}")
        print("Please update MODEL_CHECKPOINT_PATH with your actual model path")
        return None, None
    
    try:
        lit_model = DISCOLitModule.load_from_checkpoint(checkpoint_path, map_location=device)
        lit_model.eval()
        
        model = lit_model.model.to(device)
        model.eval()
        
        print(f"Model loaded successfully from {checkpoint_path}")
        print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
        
        return model, lit_model
        
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None

# Load the model
model, lit_model = load_model_from_checkpoint(MODEL_CHECKPOINT_PATH)
relative_l2_error = RelativeL2()

if model is None:
    print("\nTo use this notebook, you need to:")
    print("1. Train a model using train_combined.py")
    print("2. Update MODEL_CHECKPOINT_PATH above with your checkpoint path")

In [ ]:
train_dataset = TemporalBatchDatasetFly(
        n_batches=16,
        batch_size=BATCH_SIZE,
        sub_x=1,
        sub_t=1,
        split="train",
        input_frames=N_INPUT_FRAMES,
        output_frames=N_OUTPUT_FRAMES,
        L=16.0,
        nx=256,
        nt=100,
        T=10.0,
        fractal_power_range=(3.0, 3.001),
        fractal_degree=256,
        v_range=(0.01, 1),
        D_range=(0.001, 1),
    )

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=None, num_workers=1, prefetch_factor=1, pin_memory=False)

In [ ]:
num_integration_steps=1
N_OUTPUT_FRAMES=1

total_error = 0
test_size = 0
all_theta = []
all_theta_latent = []

for batch in tqdm(train_dataloader):
    inp, target = batch["input"], batch["target"]
    inp = inp.squeeze(1)
    target = target.squeeze(1)
            
    inp = inp.to(device)
    target = target.to(device)
    state_labels = torch.tensor([0], device=inp.device)
        
    x_shape = inp.shape
    B, T, C = x_shape[:3]
    spatial = x_shape[3:]
    dim = len(spatial)
    
    n_sample = inp.shape[0]
    #pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)
    
    #predict
    with torch.no_grad():
        # encode into 2 dimensional
        theta_latent, metadata= model.encode_theta_latent(inp, state_labels)
        theta = model.decode_theta(theta_latent, dim)
        all_theta.append(theta)
        all_theta_latent.append(theta_latent)
        n_output_frames = N_OUTPUT_FRAMES
        #pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_output_frames,)# integration_time=4/256*n_output_frames, dt=4/256, predict_normed=False, metadata=metadata)
        pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_output_frames)# integration_time=4/256, dt=1/256)#dt=4/256, predict_normed=False, metadata=metadata)

    rollout_error = relative_l2_error(pred, target[:, :n_output_frames]).item()
    total_error+=rollout_error*n_sample
    test_size+=n_sample

all_theta = torch.cat(all_theta)
all_theta_latent = torch.cat(all_theta_latent)

print('test error', total_error/test_size)

In [ ]:
print(pred.shape, target.shape)

In [ ]:
print(f"Error with encoder", rollout_error)

In [ ]:
idx=5
for t in range(n_output_frames):
    plt.plot(pred[idx].squeeze().cpu().detach()[t])

In [ ]:
for t in range(n_output_frames):
    plt.plot(target[idx].squeeze().cpu().detach()[t])

In [ ]:
# 1. OOD Dataset Loader
def load_ood_dataset(ood_type, split='train'):
  """Load specific OOD dataset"""
  dataset = TemporalBatchDatasetFly(
        n_batches=1,
        batch_size=BATCH_SIZE,
        sub_x=1,
        sub_t=1,
        split="test",
        input_frames=N_INPUT_FRAMES,
        output_frames=N_OUTPUT_FRAMES,
        L=16.0,
        nx=256,
        nt=100,
        T=10.0,
        fractal_power_range=(3.0, 3.001),
        fractal_degree=256,
        #v_range=(0.9, 1.0),
        #D_range=(0.9, 1.0),
        v_range=(0.01, 1.0),
        D_range=(0.01, 1.0),
    )
  dataloader = DataLoader(dataset, batch_size=32)
  return dataloader

# 2. Simple Direct Prediction Test
def test_direct_prediction(model, dataloader, ood_name):
  """Test standard model prediction"""
  total_error = 0
  samples = 0
  prediction = []
  for batch in dataloader:
      inp, target = batch["input"].to(device), batch["target"].to(device)
      if inp.ndim==5:
          inp = inp.squeeze(0)
          target = target.squeeze(0)
          
      print('inp', inp.shape, target.shape)
      state_labels = torch.tensor([0], device=device)

      with torch.no_grad():
          pred, _ = model(inp, state_labels, n_future_steps=target.shape[1])#, integration_time=target.shape[1])
          prediction.append(pred.cpu())
          error = relative_l2_error(pred, target).item()
          total_error += error * inp.shape[0]
          samples += inp.shape[0]

  avg_error = total_error / samples
  print(f"{ood_name} Direct Prediction Error: {avg_error:.6f}")
  return avg_error, torch.cat(prediction)

# 3. Simple Theta Prediction Test  
def test_theta_prediction(model, dataloader, ood_name):
  """Test theta-based prediction"""
  total_error = 0
  samples = 0
  prediction = []
  for batch in dataloader:
      inp, target = batch["input"].to(device), batch["target"].to(device)
      if inp.ndim==5:
          inp = inp.squeeze(0)
          target = target.squeeze(0)
      state_labels = torch.tensor([0], device=device)

      with torch.no_grad():
          # Extract theta from encoder
          theta_latent, metadata = model.encode_theta_latent(inp, state_labels)
          theta = model.decode_theta(theta_latent, dim=1)

          # Predict using theta
          pred, _ = model.solve_ode(inp[:, -1], theta, state_labels, dim=1,
                                  n_future_steps=target.shape[1], )#integration_time=target.shape[1], dt=1,
                                   #predict_normed=False, metadata={})
          prediction.append(pred.cpu())
          error = relative_l2_error(pred, target).item()
          total_error += error * inp.shape[0]
          samples += inp.shape[0]

  avg_error = total_error / samples
  print(f"{ood_name} Theta Prediction Error: {avg_error:.6f}")
  return avg_error, torch.cat(prediction)

# 4. Simple Greedy Test (using first batch only for speed)
def test_greedy_composition(model, dataloader, encoded_operators, ood_name):
    """Test greedy search on first batch"""
    first_batch = next(iter(dataloader))
    
    inp, target = first_batch["input"].to(device), first_batch["target"].to(device)
    if inp.ndim==5:
        inp = inp.squeeze(0)
        target = target.squeeze(0)
    alpha = 0 #first_batch['alpha']
    beta = 0 #first_batch['beta']
    gamma = 0# first_batch['gamma']
    theta_operators, _, metadata = encoded_operators
    pred_all = []
    for i in range(inp.shape[0]):
        #print(f"EQUATION with alpha: {alpha[i]}, beta: {beta[i]}, gamma: {gamma[i]}")
        composition, _, pred = greedy_operator_selection(model, theta_operators, inp[i:i+1], target[i:i+1], max_operators=5)
        #alphas = sum([metadata[op_id]["alpha"] for op_id in composition]).numpy()
        #betas = sum([metadata[op_id]["beta"] for op_id in composition]).numpy()
        #gammas = sum([metadata[op_id]["gamma"] for op_id in composition]).numpy()
        print(f"{ood_name} Best Greedy Composition: {composition}")
        #print(f"Parameters found: alpha={alphas}, beta={betas}, gamma={gammas}")
        print(f"\n")
        pred_all.append(pred)
    return composition, torch.cat(pred_all)

# 5. Simple Random Test  
def test_random_composition(model, dataloader, encoded_operators, ood_name, num_compositions=500, composition_lengths=[1,2,3,4,5]):
    """Test random search on first batch"""
    first_batch = next(iter(dataloader))
    inp, target = first_batch["input"].to(device), first_batch["target"].to(device)
    if inp.ndim==5:
        inp = inp.squeeze(0)
        target = target.squeeze(0)
    alpha = 0 #first_batch['alpha']
    beta = 0 #first_batch['beta']
    gamma = 0 #first_batch['gamma']
    theta_operators, _, metadata = encoded_operators
    pred_all = []
    for i in range(inp.shape[0]):
        #print(f"EQUATION with alpha: {alpha[i]}, beta: {beta[i]}, gamma: {gamma[i]}")
        composition, _, pred = random_operator_selection(model, theta_operators, inp[i:i+1], target[i:i+1], num_compositions=num_compositions, composition_lengths=composition_lengths)
        #alphas = sum([metadata[op_id]["alpha"] for op_id in composition]).numpy()
        #betas = sum([metadata[op_id]["beta"] for op_id in composition]).numpy()
        #gammas = sum([metadata[op_id]["gamma"] for op_id in composition]).numpy()
        print(f"{ood_name} Best Random Composition: {composition}")
        #print(f"Parameters found: alpha={alphas}, beta={betas}, gamma={gammas}")
        print(f"\n")
        pred_all.append(pred)
    
    return composition, torch.cat(pred_all)
    
def test_nearest_param_selection(model, dataloader, encoded_operators, ood_name, num_integration_steps=1):
      """Test nearest parameter selection on first batch"""
      first_batch = next(iter(dataloader))

      inp, target = first_batch["input"].to(device), first_batch["target"].to(device)
      if inp.ndim==5:
          inp = inp.squeeze(0)
          target = target.squeeze(0)
      alpha = 0 #first_batch['alpha']
      beta = 0  #first_batch['beta']
      gamma = 0 #first_batch['gamma']
      theta_operators, _, metadata = encoded_operators

      # Ensure theta_operators is on the right device
      theta_operators = theta_operators.to(device)

      state_labels = torch.tensor([0], device=device)
      loss_fn = RelativeL2()

      prediction = []

      for i in range(inp.shape[0]):
          print(f"EQUATION with alpha: {alpha[i]}, beta: {beta[i]}, gamma: {gamma[i]}")

          # Find nearest parameter composition
          composition = nearest_param_selection(alpha[i], beta[i], gamma[i], metadata)

          if composition:
              # Test the composition
              model.eval()
              with torch.no_grad():
                  try:
                      pred = strang_splitting_composition(
                          inp[i:i+1, -1], state_labels, composition, theta_operators, model,
                          integration_time=1.0, n_future_steps=target.shape[1], num_integration_steps=num_integration_steps
                      )
                      if pred.ndim==3:
                          pred = pred.unsqueeze(0)
                      pred = rearrange(pred, 't b c h -> b t c h')
                      prediction.append(pred)
                      test_error = loss_fn(pred, target[i:i+1]).item()

                      # Get the actual parameters of the selected operators
                      found_alphas = [metadata[op_idx]["alpha"] for op_idx in composition]
                      found_betas = [metadata[op_idx]["beta"] for op_idx in composition]
                      found_gammas = [metadata[op_idx]["gamma"] for op_idx in composition]

                      # Convert to numpy if needed
                      found_alphas = [x.numpy() if hasattr(x, 'numpy') else x for x in found_alphas]
                      found_betas = [x.numpy() if hasattr(x, 'numpy') else x for x in found_betas]
                      found_gammas = [x.numpy() if hasattr(x, 'numpy') else x for x in found_gammas]

                      print(f"{ood_name} Nearest Param Composition: {composition}")
                      print(f"Parameters found: alpha={found_alphas}, beta={found_betas}, gamma={found_gammas}")
                      print(f"Test error: {test_error:.6f}")

                  except Exception as e:
                      print(f"Error applying composition {composition}: {e}")
          else:
              print(f"No suitable operator found (all target parameters are zero)")

          print(f"\n")

      return composition , torch.cat(prediction)

# 6. Test All OOD Datasets
def test_all_ood_datasets():
  """Test all methods on all OOD datasets"""
  #ood_types = ['E_ALL', 'E_BG', 'E_ED', 'E_HE']
  #ood_types = ['E_HE']
  ood_types = ["E_ADV_DIFF"]
  results = {}
  

  for ood_type in ood_types:
      print(f"\n=== Testing {ood_type} ===")

  
      dataloader = load_ood_dataset(ood_type)
      target = []
      for batch in dataloader:
          target.append(batch['target'])
          

      # Test direct prediction
      direct_error, pred_direct = test_direct_prediction(model, dataloader, ood_type)

      # Test theta prediction  
      #theta_error, pred_theta = test_theta_prediction(model, dataloader, ood_type)

      # Test compositions (if operators available)
      if encoded_operators:
          greedy_comp, pred_greedy = test_greedy_composition(model, dataloader, encoded_operators, ood_type)
          #random_comp, pred_random = test_random_composition(model, dataloader, encoded_operators, ood_type, composition_lengths=[2,3,4])
          #nearest_comp, pred_nearest = test_nearest_param_selection(model, dataloader, encoded_operators, ood_type, num_integration_steps=10)
      else:
          greedy_comp = random_comp = None

      results[ood_type] = {
          'direct_error': direct_error,
          #'theta_error': theta_error,
          #'greedy_composition': greedy_comp,
          #'random_composition': random_comp,
          'pred_direct':pred_direct,
          "pred_random":pred_random,
          #'pred_theta':pred_theta,
          'pred_greedy':pred_greedy,
          #'pred_nearest':pred_nearest,
          'target': torch.cat(target),
      }

      #except Exception as e:
      #    print(f"Error testing {ood_type}: {e}")
      #    results[ood_type] = {'error': str(e)}

  return results

In [ ]:
#train_files = [TRAINING_FILES[key] for key in TRAINING_FILES.keys()]
theta_operators, theta_latent_operators, operator_metadata = encode_operators_from_training_data(model, train_dataloader, num_operators=128*2, n_trajectories_per_operator=1)

In [ ]:
encoded_operators = theta_operators, theta_latent_operators, operator_metadata 
#encoded_operators = all_theta, all_theta_latent, operator_metadata

In [ ]:
theta_operators.shape

In [ ]:
# Usage:
N_OUTPUT_FRAMES=34
results = test_all_ood_datasets()

In [ ]:
# HEAT + BURGERS does not work it seems

In [ ]:
# try do do nearest neighbor parameter by parameter and see if it works

In [ ]:
#results['E_BG']

In [ ]:
target = results['E_BG']['target']

In [ ]:
u_direct = results['E_BG']['pred_direct']

In [ ]:
u_composition = results['E_BG']['pred_greedy']

In [ ]:
idx=2

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(u_direct[idx].squeeze(1).cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(u_composition[idx].squeeze(1).cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(target[idx].squeeze(1).cpu().detach()[t])

## Operator Encoding

In [ ]:
def encode_operators_from_training_data(model, dataloader, num_operators=20, 
                                       n_trajectories_per_operator=4):
    """Encode operators from training trajectories."""
    if model is None:
        print("Model not loaded")
        return None
    
    print(f"Encoding {num_operators} operators from training data...")
    
    # Collect trajectories for encoding
    all_trajectories = []
    operator_metadata = []
    
    target_trajectories = n_trajectories_per_operator*num_operators
    total_collected=0
    collected = 0
    
    for batch in dataloader:
        if collected >= target_trajectories:
            break
            
        input_seq = batch['input']
        #alpha = batch['alpha']
        #beta = batch['beta']
        #gamma = batch['gamma']
        
        for sample_idx in range(input_seq.shape[0]):
            if collected >= target_trajectories:
                break
            
            trajectory = input_seq[sample_idx:sample_idx+1]
            all_trajectories.append(trajectory)
            
            # Track operator metadata
            operator_idx = collected #// n_trajectories_per_operator
            operator_metadata.append({
                'operator_id': total_collected,
                'equation_type': 0,
                'trajectory_indices': [],
                'alpha':0, #alpha[sample_idx:sample_idx+1],
                'beta':0, #beta[sample_idx:sample_idx+1],
                #'gamma':#gamma[sample_idx:sample_idx+1],
            })
            
            operator_metadata[-1]['trajectory_indices'].append(len(all_trajectories) - 1)
            collected += 1
            total_collected +=1
    
    print(f"Collected {len(all_trajectories)} trajectories for {len(operator_metadata)} operators")
    
    # Encode all trajectories
    all_theta_latent = []
    all_theta = []
    
    state_labels = torch.tensor([0], device=device)
    encoding_batch_size = 32
    
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(all_trajectories), encoding_batch_size), desc="Encoding"):
            batch_trajectories = all_trajectories[i:i+encoding_batch_size]
            batch_input = torch.cat(batch_trajectories, dim=0).to(device)
            
            theta_latent_batch, _ = model.encode_theta_latent(batch_input, state_labels)
            theta_batch = model.decode_theta(theta_latent_batch, dim=1)
            
            all_theta_latent.append(theta_latent_batch.cpu())
            all_theta.append(theta_batch.cpu())
    
    all_theta_latent = torch.cat(all_theta_latent, dim=0)
    all_theta = torch.cat(all_theta, dim=0)
    
    # Average parameters for each operator
    num_unique_operators = len(operator_metadata)
    theta_operators = torch.zeros(num_unique_operators, all_theta.shape[1])
    theta_latent_operators = torch.zeros(num_unique_operators, all_theta_latent.shape[1])
    
    for op_idx, op_meta in enumerate(operator_metadata):
        traj_indices = op_meta['trajectory_indices']
        theta_operators[op_idx] = all_theta[traj_indices].mean(dim=0)
        theta_latent_operators[op_idx] = all_theta_latent[traj_indices].mean(dim=0)
    
    print(f"\nEncoded {num_unique_operators} operators:")
    print(f"  Theta shape: {theta_operators.shape}")
    print(f"  Theta latent shape: {theta_latent_operators.shape}")
    
    # Print distribution
    eq_counts = {}
    for op_meta in operator_metadata:
        eq_type = op_meta['equation_type']
        eq_counts[eq_type] = eq_counts.get(eq_type, 0) + 1
    
    print(f"  Distribution: {eq_counts}")
    
    return theta_operators, theta_latent_operators, operator_metadata

## Operator Selection Methods

In [ ]:
def nearest_param_selection(target_alpha, target_beta, target_gamma, metadata):
  """
  For each non-zero target parameter, find the operator with the closest value.
  Combine all selected operators into a composition.
  
  Args:
      target_alpha: target alpha value (ignored if 0)
      target_beta: target beta value (ignored if 0)  
      target_gamma: target gamma value (ignored if 0)
      metadata: list/dict of operator metadata containing alpha, beta, gamma
      
  Returns:
      composition: list of operator indices (one for each non-zero target parameter)
  """
  import numpy as np

  # Convert targets to numpy if they're tensors
  if hasattr(target_alpha, 'numpy'):
      target_alpha = target_alpha.numpy()
  if hasattr(target_beta, 'numpy'):
      target_beta = target_beta.numpy()
  if hasattr(target_gamma, 'numpy'):
      target_gamma = target_gamma.numpy()

  composition = []

  # For each non-zero parameter, find the closest operator
  if target_alpha != 0:
      min_distance = float('inf')
      best_alpha_op = None

      for op_idx, op_metadata in enumerate(metadata):
          op_alpha = op_metadata['alpha']
          if hasattr(op_alpha, 'numpy'):
              op_alpha = op_alpha.numpy()

          distance = abs(target_alpha - op_alpha)
          if distance < min_distance:
              min_distance = distance
              best_alpha_op = op_idx

      if best_alpha_op is not None:
          composition.append(best_alpha_op)

  if target_beta != 0:
      min_distance = float('inf')
      best_beta_op = None

      for op_idx, op_metadata in enumerate(metadata):
          op_beta = op_metadata['beta']
          if hasattr(op_beta, 'numpy'):
              op_beta = op_beta.numpy()

          distance = abs(target_beta - op_beta)
          if distance < min_distance:
              min_distance = distance
              best_beta_op = op_idx

      if best_beta_op is not None:
          composition.append(best_beta_op)

  if target_gamma != 0:
      min_distance = float('inf')
      best_gamma_op = None

      for op_idx, op_metadata in enumerate(metadata):
          op_gamma = op_metadata['gamma']
          if hasattr(op_gamma, 'numpy'):
              op_gamma = op_gamma.numpy()

          distance = abs(target_gamma - op_gamma)
          if distance < min_distance:
              min_distance = distance
              best_gamma_op = op_idx

      if best_gamma_op is not None:
          composition.append(best_gamma_op)

  return composition

In [ ]:
def greedy_operator_selection(model, theta_operators, test_input, test_target, 
                             max_operators=5, min_improvement_threshold=5.0):
    """Greedy operator selection."""
    if model is None or theta_operators is None:
        return [], {}
    
    print(f"Running greedy operator selection...")
    print(f"Testing {theta_operators.shape[0]} operators, max length: {max_operators}")
    
    theta_operators = theta_operators.to(device)
    test_input = test_input.to(device)
    test_target = test_target.to(device)
    
    num_operators = theta_operators.shape[0]
    state_labels = torch.tensor([0], device=device)
    loss_fn = RelativeL2()
    
    # Random validation timestep
    #val_t = random.randint(0, test_input.shape[1] - 2)
    #x_val = test_input[:, val_t]
    #y_val = test_input[:, val_t + 1]
    x_val = rearrange(test_input[:, :-1], "b t c h -> (b t) c h")
    y_val = rearrange(test_input[:, 1:], "b t c h -> (b t) c h")
    
    current_composition = []
    current_best_error = float('inf')
    
    history = {
        'compositions': [],
        'errors': [],
        'method': 'greedy'
    }
    
    model.eval()
    
    for comp_length in range(1, max_operators + 1):
        print(f"\n--- Step {comp_length}: Testing all operators ---")
        
        best_error_for_step = float('inf')
        best_composition_for_step = None
        best_operator_added = None
        
        with torch.no_grad():
            for op_idx in range(num_operators):
                composition = current_composition + [op_idx]
                
                try:
                    #pred = sequential_operator_composition(
                    #    x_val, state_labels, composition, theta_operators, model,
                    #    integration_time=1.0, n_future_steps=1, num_integration_steps=1#5
                    #)
                     
                    pred = strang_splitting_composition(x_val, state_labels, composition, theta_operators, model,
                    integration_time=0.1, n_future_steps=1, num_integration_steps=1)#5)
                    
                    error = loss_fn(pred, y_val).item()
                    
                    history['compositions'].append(composition.copy())
                    history['errors'].append(error)
                    
                    if error < best_error_for_step:
                        best_error_for_step = error
                        best_composition_for_step = composition.copy()
                        best_operator_added = op_idx
                        
                except Exception as e:
                    continue
        
        if comp_length == 1:
            improvement = 0
            should_continue = True
        else:
            improvement = (current_best_error - best_error_for_step) / current_best_error * 100
            should_continue = improvement >= min_improvement_threshold
        
        print(f"  Best operator: {best_operator_added}, error: {best_error_for_step:.6f}")
        if comp_length > 1:
            print(f"  Improvement: {improvement:+.2f}%")
        
        if should_continue and best_error_for_step < current_best_error:
            current_composition = best_composition_for_step.copy()
            current_best_error = best_error_for_step
        else:
            print(f"  Stopping - insufficient improvement")
            break

    with torch.no_grad():
        pred = strang_splitting_composition(
                    test_input[:, -1], state_labels, current_composition, theta_operators, model,
                    integration_time=0.1, n_future_steps=N_OUTPUT_FRAMES, num_integration_steps=1
                )
        pred = rearrange(pred, 't b c h -> b t c h')
        test_error = loss_fn(pred, test_target).item()
                
    print(f"Greedy selection completed: {current_composition} leading to error: {test_error:.6f}")
    #print(f"\nGreedy selection completed: {current_composition} (error: {current_best_error:.6f})")
    return current_composition, history, pred

In [ ]:
def random_operator_selection(model, theta_operators, test_input, test_target, 
                             num_compositions=1000, composition_lengths=[1, 2, 3, 4, 5]):
    """Random search for operator composition."""
    if model is None or theta_operators is None:
        return [], {}
    
    print(f"Running random operator search...")
    print(f"Testing {num_compositions} random compositions")
    
    theta_operators = theta_operators.to(device)
    test_input = test_input.to(device)
    test_target = test_target.to(device)
    
    num_operators = theta_operators.shape[0]
    state_labels = torch.tensor([0], device=device)
    loss_fn = RelativeL2()
    
    val_t = random.randint(0, test_input.shape[1] - 2)
    #x_val = test_input[:, val_t]
    #y_val = test_input[:, val_t + 1]
    x_val = rearrange(test_input[:, :-1], "b t c h -> (b t) c h")
    y_val = rearrange(test_input[:, 1:], "b t c h -> (b t) c h")
    
    best_composition = []
    best_error = float('inf')
    
    history = {
        'compositions': [],
        'errors': [],
        'method': 'random',
        'total_tested': 0
    }
    
    model.eval()
    
    with torch.no_grad():
        for comp_idx in tqdm(range(num_compositions), desc="Random search"):
            comp_length = random.choice(composition_lengths)
            composition = random.sample(range(num_operators), comp_length)
            
            try:
                #pred = sequential_operator_composition(
                #    x_val, state_labels, composition, theta_operators, model,
                #    integration_time=1.0, n_future_steps=1, num_integration_steps=1#5
                #)
                
                pred = strang_splitting_composition(
                    x_val, state_labels, composition, theta_operators, model,
                    integration_time=0.1, n_future_steps=1, num_integration_steps=1#5
                )
                
                error = loss_fn(pred, y_val).item()
                
                history['compositions'].append(composition.copy())
                history['errors'].append(error)
                history['total_tested'] += 1
                
                if error < best_error:
                    best_error = error
                    best_composition = composition.copy()
                    
            except Exception as e:
                continue

    with torch.no_grad():
        pred = strang_splitting_composition(
                    test_input[:, -1], state_labels, best_composition, theta_operators, model,
                    integration_time=0.1, n_future_steps=N_OUTPUT_FRAMES, num_integration_steps=1
                )
        pred = rearrange(pred, 't b c h -> b t c h')
        test_error = loss_fn(pred, test_target).item()
                
    print(f"Random search completed: {best_composition} leading to error: {test_error:.6f}")
    print(f"Successfully tested: {history['total_tested']}/{num_compositions}")
    
    return best_composition, history, pred

In [ ]:
def compare_selection_methods(model, encoded_operators, test_sample):
    """Compare greedy vs random selection methods."""
    if model is None or encoded_operators is None:
        print("Model or operators not available")
        return {}
    
    theta_operators, theta_latent_operators, operator_metadata = encoded_operators
    
    print(f"\nComparing operator selection methods...")
    print(f"Using {theta_operators.shape[0]} operators")
    
    test_input, test_target = test_sample
    test_input = test_input.unsqueeze(0) if test_input.dim() == 3 else test_input[:1]
    test_target = test_target.unsqueeze(0) if test_target.dim() == 3 else test_target[:1]
    
    results = {}
    
    # 1. Direct prediction baseline
    print("\n--- Direct Prediction ---")
    with torch.no_grad():
        try:
            state_labels = torch.tensor([0], device=device)
            direct_pred, _ = model(
                test_input, state_labels, y=test_target,
                n_future_steps=test_target.shape[1]-1,
                integration_time=test_target.shape[1]-1
            )
            direct_error = RelativeL2()(direct_pred, test_target[:, 1:]).item()
            results['direct'] = {'error': direct_error, 'method': 'direct'}
            print(f"Direct prediction error: {direct_error:.6f}")
        except Exception as e:
            print(f"Direct prediction failed: {e}")
            results['direct'] = {'error': float('inf')}
    
    # 2. Greedy search
    print("\n--- Greedy Search ---")
    greedy_comp, greedy_hist = greedy_operator_selection(
        model, theta_operators, test_input, test_target,
        max_operators=min(5, theta_operators.shape[0])
    )
    
    if greedy_comp and greedy_hist['errors']:
        results['greedy'] = {
            'composition': greedy_comp,
            'error': min(greedy_hist['errors']),
            'evaluations': len(greedy_hist['errors']),
            'method': 'greedy'
        }
    
    # 3. Random search
    print("\n--- Random Search ---")
    random_comp, random_hist = random_operator_selection(
        model, theta_operators, test_input, test_target,
        num_compositions=min(200, theta_operators.shape[0] * 10)
    )
    
    if random_comp and random_hist['errors']:
        results['random'] = {
            'composition': random_comp,
            'error': min(random_hist['errors']),
            'evaluations': random_hist['total_tested'],
            'method': 'random'
        }
    
    # Print comparison
    print("\n" + "="*60)
    print("METHOD COMPARISON RESULTS")
    print("="*60)
    
    for method in ['direct', 'greedy', 'random']:
        if method in results and 'error' in results[method]:
            result = results[method]
            error = result['error']
            if error < float('inf'):
                comp_str = str(result.get('composition', 'N/A'))[:20]
                evals = result.get('evaluations', 'N/A')
                print(f"{method:10}: {error:.6f} | {comp_str:20} | {evals} evals")
            else:
                print(f"{method:10}: Failed")
    
    # Find best method
    valid_results = {k: v for k, v in results.items() 
                    if 'error' in v and v['error'] < float('inf')}
    
    if valid_results:
        best_method = min(valid_results.items(), key=lambda x: x[1]['error'])
        print(f"\nBest method: {best_method[0]} with error {best_method[1]['error']:.6f}")
        
        if 'direct' in valid_results:
            direct_error = valid_results['direct']['error']
            print("\nImprovement over direct prediction:")
            for method, result in valid_results.items():
                if method != 'direct' and 'composition' in result:
                    improvement = (direct_error - result['error']) / direct_error * 100
                    efficiency = improvement / result.get('evaluations', 1) * 100
                    print(f"  {method}: {improvement:+.2f}% ({efficiency:.4f}% per eval)")
    
    return results


## Test on Out-of-Distribution Data

In [ ]:
def test_operator_composition_on_data(model, encoded_operators, test_dataloader, 
                                     equation_name="Test", max_samples=3, 
                                     selection_method='greedy'):
    """Test operator composition on data."""
    if model is None or encoded_operators is None:
        return {}
    
    theta_operators, theta_latent_operators, operator_metadata = encoded_operators
    
    print(f"\nTesting {selection_method} composition on {equation_name} data...")
    print(f"Using {theta_operators.shape[0]} operators")
    
    loss_fn = RelativeL2()
    composition_results = []
    direct_results = []
    
    samples_tested = 0
    for batch_idx, batch in enumerate(test_dataloader):
        if samples_tested >= max_samples:
            break
            
        input_seq = batch['input']
        target_seq = batch['target']
        
        for sample_idx in range(min(input_seq.shape[0], max_samples - samples_tested)):
            print(f"\n--- Sample {samples_tested + 1} ---")
            
            test_input = input_seq[sample_idx:sample_idx+1].to(device)
            test_target = target_seq[sample_idx:sample_idx+1].to(device)
            
            # Direct prediction
            with torch.no_grad():
                try:
                    state_labels = torch.tensor([0], device=device)
                    direct_pred, _ = model(
                        test_input, state_labels, y=test_target,
                        n_future_steps=test_target.shape[1]-1,
                        integration_time=test_target.shape[1]-1
                    )
                    direct_error = loss_fn(direct_pred, test_target[:, 1:]).item()
                    
                    direct_results.append({
                        'sample_idx': samples_tested,
                        'error': direct_error
                    })
                    
                    print(f"  Direct error: {direct_error:.6f}")
                    
                except Exception as e:
                    print(f"  Direct prediction failed: {e}")
                    direct_error = float('inf')
            
            # Operator composition
            try:
                if selection_method == 'greedy':
                    best_composition, _ = greedy_operator_selection(
                        model, theta_operators, test_input, test_target,
                        max_operators=OPERATOR_CONFIG['max_operators']
                    )
                else:  # random
                    best_composition, _ = random_operator_selection(
                        model, theta_operators, test_input, test_target,
                        num_compositions=100
                    )
                
                if best_composition:
                    # Evaluate final composition on full sequence
                    with torch.no_grad():
                        x_test = test_input[:, -1]
                        pred_steps = []
                        current = x_test
                        
                        for step in range(test_target.shape[1]):
                            current = sequential_operator_composition(
                                current, state_labels, best_composition, 
                                theta_operators, model,
                                integration_time=1.0, n_future_steps=1,
                                num_integration_steps=1
                            )
                            pred_steps.append(current.unsqueeze(1))
                        
                        composition_pred = torch.cat(pred_steps, dim=1)
                        composition_error = loss_fn(composition_pred, test_target).item()
                        
                        improvement = (direct_error - composition_error) / direct_error * 100 if direct_error > 0 else 0
                        
                        composition_results.append({
                            'sample_idx': samples_tested,
                            'composition': best_composition,
                            'error': composition_error,
                            'improvement': improvement,
                            'method': selection_method
                        })
                        
                        print(f"  {selection_method.title()} composition: {best_composition}")
                        print(f"  Composition error: {composition_error:.6f}")
                        print(f"  Improvement: {improvement:+.2f}%")
                else:
                    print(f"  No valid composition found")
                    
            except Exception as e:
                print(f"  Composition failed: {e}")
            
            samples_tested += 1
    
    # Summary
    if composition_results and direct_results:
        comp_errors = [r['error'] for r in composition_results]
        direct_errors = [r['error'] for r in direct_results]
        improvements = [r['improvement'] for r in composition_results]
        
        print(f"\n{equation_name} Summary ({selection_method}):")
        print(f"  Samples: {samples_tested}")
        print(f"  Direct error: {np.mean(direct_errors):.6f} ± {np.std(direct_errors):.6f}")
        print(f"  Composition error: {np.mean(comp_errors):.6f} ± {np.std(comp_errors):.6f}")
        print(f"  Avg improvement: {np.mean(improvements):+.2f}%")
    
    return {
        'equation_name': equation_name,
        'method': selection_method,
        'samples_tested': samples_tested,
        'composition_results': composition_results,
        'direct_results': direct_results
    }

## Results Visualization

In [ ]:
def visualize_method_comparison(ood_test_results):
    """Visualize comparison between greedy and random methods."""
    if not ood_test_results:
        print("No results to visualize")
        return
    
    # Collect data
    greedy_data = []
    random_data = []
    
    for key, results in ood_test_results.items():
        if '_greedy' in key:
            greedy_data.extend(results['composition_results'])
        elif '_random' in key:
            random_data.extend(results['composition_results'])
    
    if not greedy_data or not random_data:
        print("Need both greedy and random results for comparison")
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Error comparison
    greedy_errors = [r['error'] for r in greedy_data]
    random_errors = [r['error'] for r in random_data]
    
    axes[0].boxplot([greedy_errors, random_errors], labels=['Greedy', 'Random'])
    axes[0].set_ylabel('Composition Error')
    axes[0].set_title('Error Distribution by Method')
    axes[0].grid(alpha=0.3)
    
    # Improvement comparison
    greedy_improvements = [r['improvement'] for r in greedy_data]
    random_improvements = [r['improvement'] for r in random_data]
    
    axes[1].boxplot([greedy_improvements, random_improvements], labels=['Greedy', 'Random'])
    axes[1].set_ylabel('Improvement over Direct (%)')
    axes[1].set_title('Improvement Distribution')
    axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.7)
    axes[1].grid(alpha=0.3)
    
    # Scatter plot
    axes[2].scatter(greedy_errors, greedy_improvements, alpha=0.7, 
                   label='Greedy', s=50)
    axes[2].scatter(random_errors, random_improvements, alpha=0.7, 
                   label='Random', s=50)
    axes[2].set_xlabel('Composition Error')
    axes[2].set_ylabel('Improvement (%)')
    axes[2].set_title('Error vs Improvement')
    axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.7)
    axes[2].legend()
    axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\nMethod Comparison Summary:")
    print("=" * 40)
    print(f"{'Method':<10} {'Avg Error':<12} {'Avg Improvement':<15} {'Samples':<8}")
    print("-" * 40)
    
    for method_name, data in [('Greedy', greedy_data), ('Random', random_data)]:
        if data:
            avg_error = np.mean([r['error'] for r in data])
            avg_improvement = np.mean([r['improvement'] for r in data])
            n_samples = len(data)
            
            print(f"{method_name:<10} {avg_error:<12.6f} {avg_improvement:<15.2f} {n_samples:<8}")

if ood_test_results:
    visualize_method_comparison(ood_test_results)
else:
    print("No results available for visualization")

print("\n" + "="*70)
print("NOTEBOOK COMPLETE")
print("="*70)
print("\nWhat you can explore:")
print("1. Modify OPERATOR_CONFIG to test different numbers of operators")
print("2. Change selection methods (greedy vs random vs exhaustive)")
print("3. Test on truly out-of-distribution data with different parameter ranges")
print("4. Analyze which operators work best for different equation types")
print("5. Try the beam search from beam_search_gpu.py for more advanced selection")

In [ ]:

# Encode operators
encoded_operators = None
if model is not None:
    available_files = {k: v for k, v in TRAINING_FILES.items() if os.path.exists(v)}
    if available_files:
        print(f"Available training files: {list(available_files.keys())}")
        encoded_operators = encode_operators_from_training_data(
            model, available_files,
            num_operators=OPERATOR_CONFIG['num_operators'],
            n_trajectories_per_operator=OPERATOR_CONFIG['n_trajectories_per_operator']
        )
    else:
        print("No training files found - please update TRAINING_FILES paths")
else:
    print("Model not loaded - skipping operator encoding")

In [ ]:

# Run method comparison if everything is available
method_comparison = {}
if model is not None and encoded_operators is not None and val_dataloaders:
    print("\n" + "="*70)
    print("COMPARING OPERATOR SELECTION METHODS")
    print("="*70)
    
    # Get test sample
    sample_eq_type = list(val_dataloaders.keys())[0]
    sample_batch = next(iter(val_dataloaders[sample_eq_type]))
    sample_input = sample_batch['input'][0]
    sample_target = sample_batch['target'][0]
    
    print(f"Using {sample_eq_type} sample for comparison...")
    
    method_comparison = compare_selection_methods(
        model, encoded_operators, (sample_input, sample_target)
    )
else:
    print("Method comparison skipped - need model, operators, and validation data")

In [ ]:

# Test on validation data as OOD example
ood_test_results = {}

if model is not None and encoded_operators is not None and val_dataloaders:
    print("\n" + "="*70)
    print("TESTING OPERATOR COMPOSITION ON OUT-OF-DISTRIBUTION DATA")
    print("="*70)
    print("Using validation data as 'OOD' example...")
    
    for eq_type, dataloader in val_dataloaders.items():
        # Test both greedy and random
        for method in ['greedy', 'random']:
            results = test_operator_composition_on_data(
                model, encoded_operators, dataloader,
                equation_name=f"{eq_type}",
                max_samples=2,
                selection_method=method
            )
            ood_test_results[f"{eq_type}_{method}"] = results
else:
    print("OOD testing skipped - need model, operators, and validation data")